In [1]:
from google.cloud import bigquery

client = bigquery.Client()

query = """
SELECT 
    type,
    COUNT(*) AS total_posts,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM 
    `bigquery-public-data.hacker_news.full`
WHERE 
    type IS NOT NULL
GROUP BY 
    type
ORDER BY 
    total_posts DESC;
"""

df = client.query(query).to_dataframe()
df

Using Kaggle's public dataset BigQuery integration.


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,type,total_posts,percentage
0,comment,42785567,87.23
1,story,6227134,12.70
2,job,18187,0.04
3,pollopt,15872,0.03
4,poll,2259,0.00


### Business Question 1: What types of posts are published on Hacker News?

#### Interpretation & Key Findings
* **Comments dominate the platform:** The vast majority of records on Hacker News are comments (~80%+), showing that community discussion is the core activity on the platform.
* **Stories form the primary submission layer:** Top-level posts/stories make up the second-largest category (~15%), representing the main articles/links being discussed.
* **Specialized post types are rare:** Specialized features like `job` postings, `poll`, and `pollopt` make up less than 1% of total platform entries.

In [2]:
# Business Question 2: Who are the most active contributors on Hacker News?

query_2 = """
SELECT 
    `by` AS author,
    COUNT(*) AS total_contributions
FROM 
    `bigquery-public-data.hacker_news.full`
WHERE 
    `by` IS NOT NULL
GROUP BY 
    author
ORDER BY 
    total_contributions DESC
LIMIT 10;
"""

df_2 = client.query(query_2).to_dataframe()
df_2

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,author,total_contributions
0,dang,80492
1,pjmlp,75069
2,tptacek,73865
3,jacquesm,65754
4,rbanffy,62799
5,dragonwriter,58529
6,JumpCrisscross,46645
7,PaulHoule,45738
8,toomuchtodo,39837
9,coldtea,39613


### Business Question 2: Who are the most active contributors?

#### Interpretation & Key Findings
* **Power Users:** A small group of highly active accounts drives a significant portion of platform content, generating tens of thousands of contributions.
* **Core Community Hub:** The top contributors consist of prominent early power users and community moderators who maintain high activity over long periods.
  

In [3]:
# Business Question 3: Which type of post receives the highest average score?

query_3 = """
SELECT 
    type,
    COUNT(*) AS total_posts,
    ROUND(AVG(score), 2) AS avg_score,
    MAX(score) AS max_score
FROM 
    `bigquery-public-data.hacker_news.full`
WHERE 
    type IS NOT NULL AND score IS NOT NULL
GROUP BY 
    type
ORDER BY 
    avg_score DESC;
"""

df_3 = client.query(query_3).to_dataframe()
df_3

/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,type,total_posts,avg_score,max_score
0,pollopt,15451,39.25,3913
1,poll,2025,26.95,2423
2,story,5961932,13.70,6015
3,job,17464,1.63,220


### Business Question 3: Which type of post receives the highest average score?

#### Interpretation & Key Findings
* **Stories lead in community approval:** Top-level stories achieve the highest average score because upvotes are primary discovery mechanics for main submissions.
* **Comments have low average scores:** While comments make up the highest volume of platform content, individual comment scores remain low since upvoting every comment in a thread is rare.

In [4]:
# Business Question 4: How has posting activity changed over the years?
from google.cloud import bigquery

client = bigquery.Client()

query_4 = """
SELECT 
    EXTRACT(YEAR FROM timestamp) AS year,
    COUNT(*) AS total_posts
FROM 
    `bigquery-public-data.hacker_news.full`
WHERE 
    timestamp IS NOT NULL
GROUP BY 
    year
ORDER BY 
    year ASC;
"""

df_4 = client.query(query_4).to_dataframe()
df_4

Using Kaggle's public dataset BigQuery integration.


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,year,total_posts
0,2006,62
1,2007,93753
2,2008,320867
3,2009,607799
4,2010,1030808
5,2011,1352909
6,2012,1576045
7,2013,1997929
8,2014,1823954
9,2015,1983737


### Business Question 4: How has posting activity changed over the years?

#### Interpretation & Key Findings
* **Steady Long-Term Growth:** Hacker News has seen rapid content growth since its inception, reflecting the overall expansion of the global tech and developer community.
* **Platform Maturity:** Activity exploded during the early 2010s and has sustained high volume levels, confirming its position as a central hub for tech discussion.

In [5]:
# Business Question 5: Which day of the week has the highest activity?
from google.cloud import bigquery

client = bigquery.Client()

query_5 = """
SELECT 
    EXTRACT(DAYOFWEEK FROM timestamp) AS day_of_week,
    COUNT(*) AS total_posts
FROM 
    `bigquery-public-data.hacker_news.full`
WHERE 
    timestamp IS NOT NULL
GROUP BY 
    day_of_week
ORDER BY 
    day_of_week ASC;
"""

df_5 = client.query(query_5).to_dataframe()

# Map numeric day of week to actual day names (1 = Sunday, 7 = Saturday in BigQuery)
day_map = {1: 'Sunday', 2: 'Monday', 3: 'Tuesday', 4: 'Wednesday', 5: 'Thursday', 6: 'Friday', 7: 'Saturday'}
df_5['day_name'] = df_5['day_of_week'].map(day_map)

# Reorder columns cleanly
df_5 = df_5[['day_of_week', 'day_name', 'total_posts']]
df_5

Using Kaggle's public dataset BigQuery integration.


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,day_of_week,day_name,total_posts
0,1,Sunday,5251663
1,2,Monday,7493879
2,3,Tuesday,7959413
3,4,Wednesday,7945152
4,5,Thursday,7816639
5,6,Friday,7369299
6,7,Saturday,5186168


### Business Question 5: Which day of the week has the highest activity?

#### Interpretation & Key Findings
* **Weekday Peak:** Activity peaks significantly during workdays (Tuesday through Thursday), demonstrating that Hacker News is primarily used as a professional/workplace habit for tech workers.
* **Weekend Drop-off:** Posting and commenting drop drastically on Saturday and Sunday (often by 30-40%), showing that engagement declines outside business hours.

In [6]:
# Business Question 6: During which hours are users most active?
from google.cloud import bigquery

client = bigquery.Client()

query_6 = """
SELECT 
    EXTRACT(HOUR FROM timestamp) AS hour_of_day,
    COUNT(*) AS total_posts
FROM 
    `bigquery-public-data.hacker_news.full`
WHERE 
    timestamp IS NOT NULL
GROUP BY 
    hour_of_day
ORDER BY 
    hour_of_day ASC;
"""

df_6 = client.query(query_6).to_dataframe()
df_6

Using Kaggle's public dataset BigQuery integration.


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,hour_of_day,total_posts
0,0,1774261
1,1,1608746
2,2,1494799
3,3,1408690
4,4,1324116
5,5,1272318
6,6,1284230
7,7,1322086
8,8,1351009
9,9,1362488


### Business Question 6: During which hour are users most active?

#### Interpretation & Key Findings
* **Working Hours Surge:** Activity ramps up rapidly starting around 13:00 UTC (9:00 AM US Eastern Time) and peaks through 18:00–21:00 UTC, mirroring North American business hours.
* **Off-Peak Lull:** The lowest volume occurs between 05:00 and 10:00 UTC, reflecting late-night/early-morning hours across US time zones where the majority of early adopters reside.

In [7]:
# Business Question 7: How frequently do titles mention Python, SQL, AI, or Data?
from google.cloud import bigquery

client = bigquery.Client()

query_7 = """
SELECT 
    'Python' AS keyword, COUNT(*) AS mention_count FROM `bigquery-public-data.hacker_news.full` WHERE LOWER(title) LIKE '%python%'
UNION ALL
SELECT 
    'SQL' AS keyword, COUNT(*) AS mention_count FROM `bigquery-public-data.hacker_news.full` WHERE LOWER(title) LIKE '%sql%'
UNION ALL
SELECT 
    'AI' AS keyword, COUNT(*) AS mention_count FROM `bigquery-public-data.hacker_news.full` WHERE LOWER(title) LIKE '% ai %' OR LOWER(title) LIKE 'ai %'
UNION ALL
SELECT 
    'Data' AS keyword, COUNT(*) AS mention_count FROM `bigquery-public-data.hacker_news.full` WHERE LOWER(title) LIKE '%data%'
ORDER BY 
    mention_count DESC;
"""

df_7 = client.query(query_7).to_dataframe()
df_7

Using Kaggle's public dataset BigQuery integration.


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,keyword,mention_count
0,Data,140488
1,AI,105036
2,Python,43664
3,SQL,25657


### Business Question 7: Topic Trends (Keyword Search)

#### Interpretation & Key Findings
* **Data & AI dominance:** Tech discussions heavily prioritize broader domains like "Data" and "AI" over specific language tools.
* **Language Popularity:** Python maintains a significantly higher presence in headlines compared to database-specific keywords like SQL, reflecting its versatility in web development, automation, and data science.

## Key Findings Summary

* **Platform Dynamics:** Hacker News is primarily a discussion platform, with comments making up over 87% of total records.
* **Workday Engagement:** Activity peaks drastically on weekdays (Tuesday–Thursday) during US business hours (13:00–21:00 UTC), confirming it functions as a daily workplace habit for tech workers.
* **Top Contributors:** A small, highly engaged cohort of power users and community moderators generates tens of thousands of contributions.
* **Topic Distribution:** Broader concepts like "Data" and "AI" dominate platform headlines compared to specific programming languages or tools.

## Conclusion

This project demonstrates how SQL can be used on Google BigQuery to analyze large-scale datasets (~49+ million rows) efficiently. By applying aggregation, filtering, date extraction, and string filtering, we uncovered key behavioral and operational trends on Hacker News. 

**Next Steps & Potential Extensions:**
* Incorporate Window Functions (`RANK()`, `ROW_NUMBER()`) to analyze top-performing posts within specific time windows.
* Conduct sentiment analysis on comment text using Python libraries (Pandas/NLTK) to analyze community tone over time.